In [ ]:
version = '6.6'

In [ ]:
from ast import literal_eval
from bs4 import BeautifulSoup
import codecs
import pandas as pd
import re
import requests

In [ ]:
def get_page_soup(url):
    response = requests.get(url)
    if response.status_code == 200:
        return BeautifulSoup(response.text, 'html.parser')

# Decode JS-style escapes (\x3d, \/)
def clean_url(raw: str) -> str:
    return codecs.decode(raw.replace(r'\/', '/'), 'unicode-escape')

def split_string_by_digits(numbered_string_to_split, pattern=r'(\d+)\.'):
    matches = list(re.finditer(pattern, numbered_string_to_split))
    result = {}

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i+1].start() if i+1 < len(matches) else len(numbered_string_to_split)
        key = int(matches[i].group(1))
        value = numbered_string_to_split[start:end].strip()
        
        if "~=" in value:
            result[key] = [item.strip() for item in value.split("~=")]
        else:
            result[key] = value[len(matches[i].group(0)):].strip()

    return result


def split_mainstats_string(mainstats_string):
    keywords = ['Sands', 'Goblet', 'Circlet']
    result = {}

    for i, key in enumerate(keywords):
        # Locate the start of the current keyword
        key_start = mainstats_string.find(key)
        if key_start == -1:
            continue  # If the keyword is not found, skip it

        # Locate the end of the section
        if i < len(keywords) - 1:
            # For Sands and Goblet, stop at the next keyword
            next_key_start = mainstats_string.find(keywords[i + 1])
            section = mainstats_string[key_start + len(key) + 1:next_key_start].strip()
        else:
            # For Circlet, stop at the first occurrence of "*", "Check Notes", or the end of the line
            end_marker_positions = [
                mainstats_string.find("*", key_start),
                mainstats_string.find("Check Notes", key_start)
            ]
            # Filter out -1 values and get the earliest valid marker
            end_marker_positions = [pos for pos in end_marker_positions if pos != -1]
            end_marker = min(end_marker_positions) if end_marker_positions else -1
            section = mainstats_string[key_start + len(key) + 1:end_marker].strip() if end_marker != -1 else mainstats_string[key_start + len(key) + 1:].strip()

        # Clean up the section and extract values
        section = section.lstrip('-').strip()  # Remove leading dashes and whitespace

        # Split values if necessary
        values = [v.strip() for v in section.split('/') if v]
        result[key] = values if len(values) > 1 else values[0]

    return result

def normalize_df(df, col_name):
    df[col_name] = df[col_name].map(lambda value: literal_eval(value))
    normalized_data = pd.json_normalize(df[col_name]).add_prefix(f'{col_name}_')
    df = pd.concat([df, normalized_data], axis=1)
    df = df.drop(col_name, axis=1)
    return df

In [ ]:
# Get list of characters from genshin.gg
url = 'https://genshin.gg/characters'
character_soup = get_page_soup(url)
char_list = []
char_object = character_soup.find_all('h2', class_='character-name')
for character in char_object:
    char_list.append(character.string.upper())

# Append additional character names where the genshin.gg name doesn't match the character title in the build sheet
char_list.extend([
    'KAMISATO AYAKA',
    'KAMISATO AYATO',
    'ARATAKI ITTO',
    'KAEDEHARA KAZUHA',
    'SANGONOMIYA KOKOMI',
    'RAIDEN SHOGUN',
    'TRAVELER',
    'SHIKANOIN HEIZOU',
    'KUJOU SARA'
])

In [ ]:
url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRq-sQxkvdbvaJtQAGG6iVz2q2UN9FCKZ8Mkyis87QHFptcOU3ViLh0_PJyMxFSgwJZrd10kbYpQFl1/pubhtml#'
content_soup = get_page_soup(url)

element_list = [
    'Pyro',
    'Hydro',
    'Electro',
    'Dendro',
    'Cryo',
    'Anemo',
    'Geo'
]

# Extract urls for each element's tab
scripts = content_soup.find_all("script")
pattern = re.compile(r'name:\s*"([^"]+)"\s*,\s*pageUrl:\s*"([^"]+)"')
element_urls = []
for script in scripts:
    if script.string and 'items.push' in script.string:
        matches = pattern.findall(script.string)
        for name, url in matches:
            if name.strip() in element_list:
                element_urls.append({'name': name.strip(), 'pageUrl': clean_url(url)})

In [ ]:
build_name_row = None
character_list = []

# The exception list below denotes characters with abnormal table header formatting before their builds
abnormal_header_characters = [
    'MAVUIKA',
    'WANDERER',
    'DURIN',
    'NICOLE'
]

# The exception list below denotes rows where a character has multiple roles whose builds share some or all relevant attributes (artifacts, main stats, substats)
# By default, we assume all the same attributes as the previous build.
exception_list_merged_cells = {
    'XIANGLING': 'OFF-FIELD DPS✩',
    'XINGQIU': 'LOW ENERGY REQUIREMENT(OFF-FIELD DPS)✩',
    'YELAN': 'LOW ENERGY REQUIREMENT(OFF-FIELD DPS)✩',
    'BEIDOU': 'OFF-FIELD DPS ✩',
    'SETHOS': 'CHARGED ATTACK DPS✩',
    'LAUMA': 'BUFF SUPPORT(LOW-ENERGY REQUIREMENT) ✩',
    'ESCOFFIER': 'LOW ENERGY REQUIREMENT (OFF-FIELD DPS)✩',
    'LAN YAN': 'DRIVER',
    'WANDERER': 'DPS (WITH EXTERNAL ATK BUFFS)✩',
    'XIANYUN': 'LOW ENERGY REQUIREMENTBUFF & HEAL SUPPORT✩'
}

for element_url in element_urls:
    content_soup = get_page_soup(element_url['pageUrl'])
    table = content_soup.find('table', class_='waffle')
    if table:
        container = table.find('tbody')

    character_name = None

    for row_index, row in enumerate(container):
        for column_index, column in enumerate(row.contents):
            # Search for character name rows
            if column.text in char_list:
                character_row = row_index
                character_name = column.text

                # Characters typically have 4 rows between the name and the first build:
                # - one additional row for the name since most names use merged double-rows
                # - one row for the last update version
                # - one row for the various column headers
                # - one row for column subheaders (WEAPON and ARTIFACT are grouped under EQUIPMENT, MAIN STATS and SUBSTATS are grouped under ARTIFACT STATS
                # As of 2026-05-26, the exceptions are Mavuika, Durin, and Nicole, who only use one row for their names, and Wanderer, who is missing the column header row for some reason
                if character_name in abnormal_header_characters:
                    build_name_row = character_row + 4
                else:
                    build_name_row = character_row + 5
            # Once we reach the NOTES row, we stop
            elif column.text[:5].upper() == 'NOTES':
                build_name_row = None
                break
            elif row_index == build_name_row:
                column_text = column.text
                # Wanderer's formatting is all over the place, so aside from missing a header row that other characters have,
                # he also seems to have an additional column that shifts the column indices of his first build one higher compared to everyone else,
                # but not those of his second.
                if character_name == 'WANDERER' and not build_name == 'DPS (WITH EXTERNAL ATK BUFFS)✩':
                    match column_index:
                        case 2:
                            # For his first build, the build_name will be set to empty by this case, then set to the actual value in the next case
                            # For his second build, the build_name will be set by this case.
                            # The build_name check a few lines before will then prevent the other cases from being hit, causing the same attribute values to be inherited from the first build.
                            build_name = column_text.lstrip()
                        # The remaining cases are only triggered for his first build
                        case 3:
                            build_name = column_text.lstrip()
                        case 4:
                            weapons = split_string_by_digits(column_text)
                        case 5:
                            artifacts = split_string_by_digits(column_text)
                        case 6:
                            main_stats = split_mainstats_string(column_text)
                        case 7:
                            substats = split_string_by_digits(column_text)
                elif not (character_name in exception_list_merged_cells.keys() and exception_list_merged_cells[character_name] == build_name):
                    match column_index:
                        case 2:
                            build_name = column_text.lstrip()
                        case 3:
                            weapons = split_string_by_digits(column_text)
                        case 4:
                            artifacts = split_string_by_digits(column_text)
                        case 5:
                            main_stats = split_mainstats_string(column_text)
                        case 6:
                            substats = split_string_by_digits(column_text)
                else:
                    # We perform special handling for characters whose builds don't use all the same attributes as the previous build here:
                    match column_index:
                        case 3:
                            # Strictly speaking, while some of the characters in the exception list have differing weapons between builds,
                            # we don't really care about weapons for sake of the spreadsheet, so we don't bother to actually perform any special handling
                            pass
                        case 4:
                            # Characters with different artifact sets between builds
                            if character_name == 'SETHOS' and build_name == 'CHARGED ATTACK DPS✩':
                                artifacts = split_string_by_digits(column_text)
                        case 5:
                            # Characters with different main stats between builds
                            if ((character_name == 'LAUMA' and build_name == 'BUFF SUPPORT(LOW-ENERGY REQUIREMENT) ✩')
                                    or (character_name == 'ESCOFFIER' and build_name == 'LOW ENERGY REQUIREMENT (OFF-FIELD DPS)✩')
                                    or (character_name == 'LAN YAN' and build_name == 'DRIVER')):
                                main_stats = split_mainstats_string(column_text)
                        case 6:
                            # Characters with different substats between builds
                            pass
                
        # By this point, we should have enumerated through all the columns, so if we're in a build_name_row, we can save the results
        if row_index == build_name_row:
            character_dict = {
                'character_name': character_name,
                'element': element_url['name'],
                'build_name': build_name,
                'weapons': weapons,
                'artifacts': artifacts,
                'main_stats': main_stats,
                'substats': substats
            }
            character_list.append(character_dict)
            build_name_row += 1

In [ ]:
df = pd.DataFrame.from_dict(character_list)
df.to_csv(f'gi_rsh_output_{version}.csv', index=False)
df

In [ ]:
df = pd.read_csv(f'gi_rsh_output_{version}.csv')

for item in ['artifacts', 'main_stats', 'substats']:
    df = normalize_df(df, item)

# Some characters have additional rows with notes about their desired stats, which manifest as additional builds.
# We filter these out by excluding builds with names that are not all upper case.
df = df[df.apply(lambda build: build['build_name'].isupper(), axis=1)]

# Add new column for prioritized builds
df['is_priority_build'] = df['build_name'].apply(lambda build_str: 1 if "✩" in str(build_str) else 0)

df

In [ ]:
replacement_for_placeholders = {
    '80 EM set': "Wanderer's Troupe, Gilded Dreams, Flower of Paradise Lost, Aubade of Morningstar and Moon, Instructor",
    '18% ATK set': "Gladiator's Finale, Shimenawa's Reminiscence, Vermillion Hereafter, Echoes of an Offering, Nighttime Whispers in the Echoing Woods, Fragment of Harmonic Whimsy, Unfinished Reverie, A Day Carved From Rising Winds",
    '20% HP set': "Tenacity of the Millelith, Vourukasha's Glow",
    '15% Healing Bonus set': "Ocean-Hued Clam, Song of Days Past",
    '20% Energy Recharge set': "Emblem of Severed Fate, Silken Moon's Serenade, The Exile"
}

crit_patterns = [
    'Crit Rate / DMG',
    'CRIT Rate / DMG',
    'Crit Rate/DMG',
    'CRIT Rate/DMG',
    'Crit Rate, DMG',
    'CRIT Rate, DMG',
    'Crit Rate,DMG',
    'CRIT Rate,DMG',
]

crit_patterns_with_trailing_commas = [
    'Crit,',
    'CRIT,'
]

col_list = [col_name for col_name in df.columns if 'artifacts' in col_name or 'main_stats' in col_name or 'substats' in col_name]
for col in col_list:
    df[col] = df[col].apply(lambda col_value: ', '.join(col_value) if isinstance(col_value, list) else col_value)

    # Replace string variants with standardized values
    for key, value in replacement_for_placeholders.items():
        df[col] = df[col].str.replace(key, value)
    for crit_pattern in crit_patterns:
        df[col] = df[col].str.replace(crit_pattern, 'Crit Rate / Crit DMG')
    for crit_pattern in crit_patterns_with_trailing_commas:
        df[col] = df[col].str.replace(crit_pattern, 'Crit Rate / Crit DMG,')

artifact_cols = [col_name for col_name in df.columns.tolist() if 'artifacts' in col_name]
substat_cols = [col_name for col_name in df.columns.tolist() if 'substats' in col_name]

# Add new columns for spreadsheet calculations
df['substats'] = df[substat_cols].apply(lambda row: ', '.join([str(x) for x in row if pd.notna(x)]), axis=1)
df['substats'] = df['substats'].str.replace('Atk%', 'ATK%')
df['substats'] = df['substats'].str.replace('ER%', 'Energy Recharge')

df['artifacts_concat'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 4)) = 0), "", TEXTJOIN(", ", TRUE, INDIRECT(ADDRESS(ROW(), COLUMN() + 1)):INDIRECT(ADDRESS(ROW(), COLUMN() + START!$B$4))))'
df['substats_if'] = '=IF(AND(START!$B$3=1, INDIRECT(ADDRESS(ROW(), COLUMN()- 14)) = 0), "", INDIRECT(ADDRESS(ROW(), COLUMN()+1)))'

df

In [ ]:
# Reorder columns
new_col_order = ['character_name', 'element', 'build_name', 'is_priority_build', 'main_stats_Sands', 'main_stats_Goblet',
                 'main_stats_Circlet', 'artifacts_concat']
new_col_order.extend(artifact_cols)
new_col_order.append('substats_if')
new_col_order.append('substats')
new_col_order.extend(substat_cols)

df = df[new_col_order]
df

In [ ]:
# Export final csv
df.to_csv(f'gi_rsh_output_full_{version}.csv', index=False)